#### Imports & Setup

In [4]:
import sys
sys.path.append('.')

import os
import mlflow
import dagshub
import pandas as pd
import numpy as np
from Models.preprocessing import load_data, load_test_data, FraudPreprocessor, ColumnSelector

from dotenv import load_dotenv; load_dotenv()
dagshub.init(repo_owner="lchit22", 
             repo_name="ml-assignment-fraud-detection", 
             mlflow=True)

Initialized MLflow to track repo "lchit22/ml-assignment-fraud-detection"

Repository lchit22/ml-assignment-fraud-detection initialized!

#### Load Best Model from Registry

In [11]:
import glob, yaml, pickle, shutil

# LoggedModel artifact location (from MLmodel file)
LOGGED_MODEL_URI = "mlflow-artifacts:/6238dd5f86534c6591e0ebebce442d23/models/m-18d26d68670149c9ba133052d96418fa/artifacts"

shutil.rmtree('./model_cache', ignore_errors=True)
local_dir = mlflow.artifacts.download_artifacts(
    artifact_uri=LOGGED_MODEL_URI,
    dst_path='./model_cache'
)
print(f"Downloaded to: {local_dir}")
print(f"Contents: {os.listdir(local_dir)}")

# Find and load the pickle directly (avoids Windows path/URI bug)
mlmodel_files = glob.glob(os.path.join(local_dir, '**', 'MLmodel'), recursive=True)
if not mlmodel_files:
    raise FileNotFoundError(f"MLmodel not found. Contents: {os.listdir(local_dir)}")

model_dir = os.path.dirname(mlmodel_files[0])
with open(mlmodel_files[0]) as f:
    mlmodel_cfg = yaml.safe_load(f)

pkl_file = mlmodel_cfg['flavors']['sklearn']['pickled_model']
with open(os.path.join(model_dir, pkl_file), 'rb') as f:
    model = pickle.load(f)

print(f"Loaded: {type(model)}")

Downloaded to: C:\Users\Likun\OneDrive\Desktop\Machine Learning\ml-assignment-fraud-detection\model_cache\artifacts
Contents: ['conda.yaml', 'MLmodel', 'model.pkl', 'python_env.yaml', 'requirements.txt']
Loaded: <class 'sklearn.pipeline.Pipeline'>


#### Load & Prepare Test Data

In [12]:
# load raw test data — NO manual preprocessing needed
# the pipeline handles everything internally
test_df = load_test_data()

# keep TransactionID separate for submission file
transaction_ids = test_df['TransactionID']

# drop TransactionID before passing to pipeline
X_test = test_df.drop(columns=['TransactionID'])

print(f"Test transactions: {len(X_test):,}")
print(f"Test features:     {X_test.shape[1]}")

Test transactions: 506,691
Test features:     432


#### Generate Predictions

In [13]:
predictions = model.predict_proba(X_test)[:, 1]

print(f"Predictions shape: {predictions.shape}")
print(f"Min:  {predictions.min():.4f}")
print(f"Max:  {predictions.max():.4f}")
print(f"Mean: {predictions.mean():.4f}")

C:\Users\Likun\OneDrive\Desktop\Machine Learning\ml-assignment-fraud-detection\Models\preprocessing.py:75: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['hour']                   = X['TransactionDT'] % 86400 // 3600
C:\Users\Likun\OneDrive\Desktop\Machine Learning\ml-assignment-fraud-detection\Models\preprocessing.py:76: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['day_of_week']            = X['TransactionDT'] // 86400 % 7
C:\Users\Likun\OneDrive\Desktop\Machine Learning\ml-assignment-fraud-detection\Models\preprocessing.p

Predictions shape: (506691,)
Min:  0.0000
Max:  1.0000
Mean: 0.0252


#### Create Submission File

In [14]:
submission = pd.DataFrame({
    'TransactionID': transaction_ids,
    'isFraud':       predictions
})

submission.to_csv('submission.csv', index=False)

print("submission.csv saved")
print(submission.head(10))

submission.csv saved
   TransactionID   isFraud
0        3663549  0.000051
1        3663550  0.000013
2        3663551  0.000689
3        3663552  0.000171
4        3663553  0.000549
5        3663554  0.001205
6        3663555  0.000655
7        3663556  0.002912
8        3663557  0.000031
9        3663558  0.002385
